In [1]:
# Copy code files into current directory from drive.

import shutil
import os
import sys

!python -c "import biosppy;" || !pip install biosppy
!python -c "import wfdb;" || !pip install wfdb

WANDB_PROJECT_NAME = "ECG Experiments"


/bin/bash: /home/vlermakov/miniconda3/envs/ecg/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/vlermakov/miniconda3/envs/ecg/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [4]:
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))

[]


2023-09-22 13:02:11.626930: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Set MITBIH_RUN to ***False*** to test on **UCSD** data and to ***True*** to test on **MIT-BIH** Dataset

In [2]:
# Test on UCSD data (if set to False) 
MITBIH_RUN = False

run_parameters = {}

if(MITBIH_RUN):
    run_parameters["csv_file_name"] = 'mit_bih_dataset.csv'
    run_parameters['EVALUATE_TRIGGER_DETECTIONS'] = False
    
else:
    run_parameters["csv_file_name"] = 'training_and_evaluation_dataset.csv'
    run_parameters['EVALUATE_TRIGGER_DETECTIONS'] = True

run_parameters['evaluation_type'] = 'test'
run_parameters['EVALUATE_HAMILTON'] = True
run_parameters['EVALUATE_CNN_PREDICTOR'] = True
run_parameters['EVALUATE_CNN_DETECTOR'] = True

run_parameters["DETECTION_CONFIDENCE_THRESHOLD"] = 30.0,
run_parameters["PEAK_COOLDOWN_THRESHOLD"] = 50.0,


In [3]:
from ucsd_ecg_dataset import ECGDataset
from deep_qrs_detector import DeepQRSDetector
from deep_qrs_predictor import DeepQRSPredictor
from ecg_dataset_manager import DatasetManager

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import biosppy.signals as bsp

# The confusion matrix calculation
import ja_analysis

DATASET_SAMPLING_RATE = 1000
TARGET_SAMPLING_RATE = 250

UCSD_PROCESSED_DATASET_LOCATION = './data/preprocessed/'
MITDB_DATASET_LOCATION = './data/mitdb/raw/'

dm = DatasetManager(UCSD_PROCESSED_DATASET_LOCATION,'',MITDB_DATASET_LOCATION)


def calculateSensitivity(conf_matrix):
    return (conf_matrix['TP'] / (conf_matrix['TP'] + conf_matrix['FN'] + 0.00001))

def calculatePPV(conf_matrix):
    return (conf_matrix['TP'] / (conf_matrix['TP'] + conf_matrix['FP'] + 0.00001))

def calculateFPR(conf_matrix):
    return (conf_matrix["FP"]/(conf_matrix["TN"] + conf_matrix["FP"] + 0.00001))

def calculateSpecificity(conf_matrix):
    return (conf_matrix["TN"] / (conf_matrix["FP"] + conf_matrix["TN"] + 0.00001))

def calculateF1(conf_matrix):
    sensitivity = calculateSensitivity(conf_matrix)
    ppv = calculatePPV(conf_matrix)
    return (2* sensitivity*ppv) / (sensitivity+ppv + 0.00001)


2023-09-22 12:48:28.660534: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-22 12:48:29.264294: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [21]:
import pandas as pd

# If we are evaluating VCG trigger, get the detections from the dataset
if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
    ucsd_ds = ECGDataset("/media/vlermakov/data/UCSD_ECG_SOURCE_DATA")

cnn_detector = DeepQRSDetector('./models/cnn_detector.h5')
cnn_predictor = DeepQRSPredictor('./models/cnn_predictor.h5',run_parameters['DETECTION_CONFIDENCE_THRESHOLD'],run_parameters['PEAK_COOLDOWN_THRESHOLD'])

dataset_df = pd.read_csv(os.path.join('./data/',run_parameters['csv_file_name']))

allTriggerResults = []
allHamiltonResults = []
allCnnDetectorResults = []
allCnnPredictorResults = []

results_dataframe = pd.DataFrame(columns=["detection_type","TP","TN","FP","FN","jitter","accuracy","ja_score","MAE","offset","offset_ms","loss","sensitivity","specificity","ppv", "fpr", "F1"])

for index, row in dataset_df.iterrows():
    minimum = 0
    maximum = 240000000

    if(row['scan'] is None):
      continue

    if(row['type'] != run_parameters['evaluation_type']):
        continue

    print(f"Processing {row['scan']} {row['scan']}")

    if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
        try:
          trigger_detections = ucsd_ds.get_trigger_detections(row['location'],row['timestamp'],int(row['session_id']))
          trigger_detections = trigger_detections / 4
        except:
          trigger_detections = []
    #try:
    ecg2_downsampled_data, labelstudio_annotations = dm.LoadSignalAndAnnotations(row['scan'], database = row['dataset'])
#     except:
#         continue

    labelstudio_annotations.sort()

    if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
        if(len(trigger_detections) == 0):
            print("No trigger detections found for " + row['scan'])
            continue
        ja_result_trigger = ja_analysis.evaluate(np.array(trigger_detections), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"Trigger","TP":ja_result_trigger["TP"],"TN":ja_result_trigger["TN"],"FP":ja_result_trigger["FP"],"FN":ja_result_trigger["FN"],"jitter":ja_result_trigger["ja"],"accuracy":ja_result_trigger["accuracy"],"ja_score":ja_result_trigger["ja"],"MAE":ja_result_trigger["MAE"],"offset":512,"offset_ms":0,'sensitivity': calculateSensitivity(ja_result_trigger),'specificity': calculateSpecificity(ja_result_trigger),'ppv' : calculatePPV(ja_result_trigger), 'fpr' : calculateFPR(ja_result_trigger), 'F1' : calculateF1(ja_result_trigger)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allTriggerResults.append(ja_result_trigger)

    if(run_parameters['EVALUATE_HAMILTON']):
        biosspy_peaks = bsp.ecg.hamilton_segmenter(ecg2_downsampled_data, sampling_rate=250)
        biosspy_peaks = biosspy_peaks[0]

        biosspy_peaks.sort()

        biosppy_results = ja_analysis.evaluate(np.array(biosspy_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"Hamilton","TP":biosppy_results["TP"],"TN":biosppy_results["TN"],"FP":biosppy_results["FP"],"FN":biosppy_results["FN"],"jitter":biosppy_results["ja"],"accuracy":biosppy_results["accuracy"],"ja_score":biosppy_results["ja"],"MAE":biosppy_results["MAE"],"offset":512,"offset_ms":0,'sensitivity': calculateSensitivity(biosppy_results),'specificity': calculateSpecificity(biosppy_results),'ppv' : calculatePPV(biosppy_results), 'fpr' : calculateFPR(biosppy_results), 'F1' : calculateF1(biosppy_results)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)
        allHamiltonResults.append(biosppy_results)

    if(run_parameters['EVALUATE_CNN_DETECTOR']):
        # Detect R peaks
        cnn_detector_r_peaks,mean_detection_offset = cnn_detector.detect_peaks(ecg2_downsampled_data)


        # Sort the cnn_detector_r_peaks list
        cnn_detector_r_peaks.sort()

        ja_result_cnn_detector = ja_analysis.evaluate(np.array(cnn_detector_r_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"CNN Detector","TP":ja_result_cnn_detector["TP"],"TN":ja_result_cnn_detector["TN"],"FP":ja_result_cnn_detector["FP"],"FN":ja_result_cnn_detector["FN"],"jitter":ja_result_cnn_detector["ja"],"accuracy":ja_result_cnn_detector["accuracy"],"ja_score":ja_result_cnn_detector["ja"],"MAE":ja_result_cnn_detector["MAE"],"offset":mean_detection_offset,"offset_ms":4.0*(mean_detection_offset-512.0),'sensitivity': calculateSensitivity(ja_result_cnn_detector),'specificity': calculateSpecificity(ja_result_cnn_detector),'ppv' : calculatePPV(ja_result_cnn_detector), 'fpr' : calculateFPR(ja_result_cnn_detector), 'F1' : calculateF1(ja_result_cnn_detector)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allCnnDetectorResults.append(ja_result_cnn_detector)

    if(run_parameters['EVALUATE_CNN_PREDICTOR']):
        # Detect R peaks
        cnn_predictor_r_peaks,mean_prediction_offset = cnn_predictor.detect_peaks(ecg2_downsampled_data)


        # Sort the cnn_predictor_r_peaks list
        cnn_predictor_r_peaks.sort()

        ja_result_cnn_predictor = ja_analysis.evaluate(np.array(cnn_predictor_r_peaks), np.array(labelstudio_annotations), 250.0, len(ecg2_downsampled_data)) # perform interval based analysis
        
        # Add the results to the dataframe
        new_row = {"scan":row['scan'],"detection_type":"CNN Predictor","TP":ja_result_cnn_predictor["TP"],"TN":ja_result_cnn_predictor["TN"],"FP":ja_result_cnn_predictor["FP"],"FN":ja_result_cnn_predictor["FN"],"jitter":ja_result_cnn_predictor["ja"],"accuracy":ja_result_cnn_predictor["accuracy"],"ja_score":ja_result_cnn_predictor["ja"],"MAE":ja_result_cnn_predictor["MAE"],"offset":mean_prediction_offset,"offset_ms":4.0*(mean_prediction_offset-512.0),'sensitivity': calculateSensitivity(ja_result_cnn_predictor),'specificity': calculateSpecificity(ja_result_cnn_predictor),'ppv' : calculatePPV(ja_result_cnn_predictor), 'fpr' : calculateFPR(ja_result_cnn_predictor), 'F1' : calculateF1(ja_result_cnn_predictor)}

        new_df = pd.DataFrame([new_row])
        results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)

        allCnnPredictorResults.append(ja_result_cnn_predictor)


# Print the results
if(run_parameters['EVALUATE_TRIGGER_DETECTIONS']):
    trigger_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['accuracy'].mean()
    trigger_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['ja_score'].mean()
    trigger_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'Trigger']['F1'].mean()

    print("Trigger Detection Results: {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(trigger_average_f1*100, trigger_average_accuracy * 100, trigger_average_jitter * 1000,trigger_average_jitter))
if(run_parameters['EVALUATE_HAMILTON']):
    hamilton_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['accuracy'].mean()
    hamilton_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['ja_score'].mean()
    hamilton_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'Hamilton']['F1'].mean()

    print("Hamilton Detection Results: {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(hamilton_average_f1 * 100,hamilton_average_accuracy * 100, hamilton_average_jitter * 1000,hamilton_average_jitter))


ecg_detector_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['accuracy'].mean()
ecg_detector_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['ja_score'].mean()
ecg_detector_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Detector']['F1'].mean()

print("CNN Detector Results:  {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(ecg_detector_average_f1 * 100,ecg_detector_average_accuracy * 100, ecg_detector_average_jitter * 1000,ecg_detector_average_jitter))

ecg_predictor_average_accuracy = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['accuracy'].mean()
ecg_predictor_average_jitter = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['ja_score'].mean()
ecg_predictor_average_f1 = results_dataframe.loc[results_dataframe['detection_type'] == 'CNN Predictor']['F1'].mean()

print("CNN Predictor Results:  {:.2f}% F1, {:.2f}% accuracy, {:.2f} ms jitter. JA: {:.2f}".format(ecg_predictor_average_f1 * 100,ecg_predictor_average_accuracy * 100, ecg_predictor_average_jitter * 1000,ecg_predictor_average_jitter))

# Save the results to a CSV file
results_dataframe.to_csv("evaluation_results.csv")

Processing hillcrestmr_0103202313_46_05_99 hillcrestmr_0103202313_46_05_99
Trigger file:  /media/vlermakov/data/UCSD_ECG_SOURCE_DATA/hillcrestmr/ECG3Trig_fgre_0103202313_46_05_99
{'MAE': 0.002306122448979592, 'jitter': 0.001, 'TP': 49, 'TN': 90, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.9090909090909091}
{'MAE': 0.023346938775510202, 'jitter': 0.0, 'TP': 49, 'TN': 90, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 1.0}
{'MAE': 0.009580038265306122, 'jitter': 0.0021875, 'TP': 49, 'TN': 90, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.8205128205128205}


/tmp/ipykernel_97035/4199516589.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_dataframe = pd.concat([results_dataframe, new_df], ignore_index=True)


{'MAE': 0.03233333333333333, 'jitter': 0.02, 'TP': 48, 'TN': 90, 'FP': 0, 'FN': 1, 'accuracy': 0.9928057553956835, 'ja': 0.33093525179856115}
Processing hillcrestmr_0103202316_44_26_651 hillcrestmr_0103202316_44_26_651
Trigger file:  /media/vlermakov/data/UCSD_ECG_SOURCE_DATA/hillcrestmr/ECG3Trig_fgre_0103202316_44_26_651
{'MAE': 0.022375000000000003, 'jitter': 0.003999999999999999, 'TP': 32, 'TN': 107, 'FP': 13, 'FN': 0, 'accuracy': 0.9144736842105263, 'ja': 0.6531954887218046}
{'MAE': 0.018000000000000002, 'jitter': 0.004, 'TP': 32, 'TN': 108, 'FP': 12, 'FN': 0, 'accuracy': 0.9210526315789473, 'ja': 0.6578947368421053}
{'MAE': 0.00586279296875, 'jitter': 0.0029374999999999996, 'TP': 32, 'TN': 119, 'FP': 1, 'FN': 0, 'accuracy': 0.993421052631579, 'ja': 0.767861683193491}
{'MAE': 0.033419354838709676, 'jitter': 0.012, 'TP': 31, 'TN': 118, 'FP': 2, 'FN': 1, 'accuracy': 0.9802631578947368, 'ja': 0.44557416267942584}
Processing hillcrestmr_0124202315_09_13_486 hillcrestmr_0124202315_09_13

{'MAE': 0.06352380952380952, 'jitter': 0.028, 'TP': 42, 'TN': 140, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.2631578947368421}
Processing mractri_1214202209_38_05_434 mractri_1214202209_38_05_434
Trigger file:  /media/vlermakov/data/UCSD_ECG_SOURCE_DATA/mractri/ECG3Trig_fgre_1214202209_38_05_434
{'MAE': 0.0022121212121212126, 'jitter': 0.001, 'TP': 33, 'TN': 112, 'FP': 0, 'FN': 1, 'accuracy': 0.9931506849315068, 'ja': 0.9028642590286425}
{'MAE': 0.02388235294117647, 'jitter': 0.004, 'TP': 34, 'TN': 112, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.7142857142857143}
{'MAE': 0.00337454044117647, 'jitter': 0.0017812499999999998, 'TP': 34, 'TN': 112, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.8488063660477453}
{'MAE': 0.07858823529411765, 'jitter': 0.032, 'TP': 34, 'TN': 112, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.23809523809523808}
Processing mractri_0209202308_35_58_378 mractri_0209202308_35_58_378
Trigger file:  /media/vlermakov/data/UCSD_ECG_SOURCE_DATA/mractri/ECG3Trig_fgre_0209202

{'MAE': 0.008709635416666665, 'jitter': 0.001984375, 'TP': 36, 'TN': 112, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.8344198174706648}
{'MAE': 0.024111111111111118, 'jitter': 0.014, 'TP': 36, 'TN': 112, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.4166666666666667}
Processing thorntonmr_1215202212_43_26_165 thorntonmr_1215202212_43_26_165
Trigger file:  /media/vlermakov/data/UCSD_ECG_SOURCE_DATA/thorntonmr/ECG3Trig_fgre_1215202212_43_26_165
{'MAE': 0.0018571428571428571, 'jitter': 0.001, 'TP': 42, 'TN': 101, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.9090909090909091}
{'MAE': 0.0020952380952380953, 'jitter': 0.0, 'TP': 42, 'TN': 101, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 1.0}
{'MAE': 0.002953125, 'jitter': 0.0011171875000000001, 'TP': 42, 'TN': 101, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.8995080815179198}
{'MAE': 0.024666666666666667, 'jitter': 0.015999999999999997, 'TP': 42, 'TN': 101, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.3846153846153847}
Processing thorntonmr_1215202219_0

{'MAE': 0.0025994318181818184, 'jitter': 0.0013124999999999999, 'TP': 33, 'TN': 118, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.8839779005524862}
{'MAE': 0.013696969696969697, 'jitter': 0.004, 'TP': 33, 'TN': 118, 'FP': 0, 'FN': 0, 'accuracy': 1.0, 'ja': 0.7142857142857143}
Trigger Detection Results: 96.42% F1, 97.71% accuracy, 816.21 ms jitter. JA: 0.82
Hamilton Detection Results: 96.30% F1, 98.08% accuracy, 778.65 ms jitter. JA: 0.78
CNN Detector Results:  99.33% F1, 99.72% accuracy, 816.73 ms jitter. JA: 0.82
CNN Predictor Results:  97.42% F1, 98.80% accuracy, 468.88 ms jitter. JA: 0.47


In [22]:
results_dataframe.groupby('detection_type')[['accuracy','ppv','fpr','F1','MAE']].mean()

,accuracy,ppv,fpr,F1,MAE
detection_type,,,,,
CNN Detector,0.997179,0.994182,0.002003,0.993264,0.007234
CNN Predictor,0.988032,0.985485,0.004186,0.974163,0.033776
Hamilton,0.980776,0.941074,0.023159,0.962982,0.023321
Trigger,0.977062,0.953685,0.025990,0.964182,0.008444
